<div class='bar_title'></div>

*Simulation for Decision Making (S4DM)*

# Simulation Modeling Using Simpy (Part 2)

Gunther Gust & Govind Rao <br>
Chair for Enterprise AI <br>
Data Driven Decisions Group <br>
Center for Artificial Intelligence and Data Science (CAIDAS)


<img src="images/d3.png" style="width:20%; float:left;" />

<img src="images/CAIDASlogo.png" style="width:20%; float:left;" />

# Today: Model Translation

<img src="images/simulation_study_steps_translation.png" style="width:100%; float:left;" />

This lecture continues with the **Model Translation** step — we introduce modeling conventions and object-oriented programming to structure our SimPy simulations systematically.

- **Input:** A conceptual model (e.g., "students arrive, queue, get served")
- **Output:** A running, well-structured computer simulation using SimPy

# Last lecture: 

* What is discrete-event simulation?
* First steps in SimPy:
    * Processes and the event loop (`yield`, `env.timeout`)
    * Process interactions (example: driving and charging)
    * Shared resources (example: requesting a spot in a charging station)
    * Three types of `yield`: timeout, process, resource request

# Agenda for today

* Object-oriented programming (OOP): The basics you need
* Modeling Conventions
* Application in Carwash Example
* Data Collection with an `EventLogger`


Credits: The following content is adapted from the [simpy documentation](https://simpy.readthedocs.io/en/latest/examples/carwash.html) 

# Background: Object-Oriented Programming (OOP)

Our modeling conventions rely on **classes** — the central concept of object-oriented programming. Here's a minimal overview of what you need:

| OOP concept | What it means | Example |
|---|---|---|
| **Class** | A blueprint for creating objects | `class Car:` |
| **Instance** | A concrete object created from a class | `car1 = Car(env, "Car 1")` |
| **Attribute** | A variable that belongs to an instance | `self.name`, `self.env` |
| **Method** | A function defined inside a class | `def run(self, carwash):` |
| **`__init__`** | The constructor — called when an instance is created | `def __init__(self, env, name):` |
| **`self`** | Refers to the current instance inside a method | `self.env = env` |

**Key idea:** A class bundles **data** (attributes) and **behavior** (methods) together. We can create multiple instances of the same class, each with its own data.

```python
class Car:
    def __init__(self, env, name):   # constructor: called when Car(...) is created
        self.env = env               # store the environment as an attribute
        self.name = name             # store the name as an attribute

    def run(self, carwash):          # method: defines what the car does
        ...
```

You don't need deep OOP knowledge for this course — the table above covers everything we use. For further practice, see the resources at the end of this notebook.

# Modeling Conventions

In the last lecture, you saw SimPy code written as simple functions and classes. Now we introduce a **convention** that gives every simulation model the same structure. This makes models easier to read, debug, and extend.

| Component | Convention | Implementation |
|---|---|---|
| **Entity** (e.g. customer, car) | Python class | `class Car:` |
| **Entity flow** through the system | Method within the entity class | `def run(self, carwash):` |
| **Entity generation** (arrivals) | Function outside the entity class | `def car_generator(env, ...):` |
| **Resource** (e.g. machine, counter) | Python class | `class Carwash:` |
| **Service** the resource provides | Method within the resource class | `def wash(self, car):` |
| **Data collection** | Dedicated logger class | `class EventLogger:` |

This convention is not mandatory for SimPy, but it helps to structure code consistently. We will use it throughout the course. Let's see each component in action with a **carwash example**.

## Carwash Example

A carwash with a limited number of machines. Cars arrive, wait for a free machine, get washed, and leave. We'll build this step by step, following the modeling convention:

1. **Resource** → `Carwash` class (machines + wash service)
2. **Entity** → `Car` class (flow: arrive → wait → wash → leave)
3. **Entity generator** → `car_generator()` function (arrivals over time)

In [1]:
import random
import simpy

### Step 1: Resource (`Carwash` class)

You already know `simpy.Resource` from the last lecture. The new idea here is to **wrap it in a domain class** that also defines the **service** it provides:

In [2]:
class Carwash: #define a separate class for the resource
    """A carwash has a limited number of machines (``NUM_MACHINES``) to
    clean cars in parallel.

    Cars have to request one of the machines. When they got one, they
    can start the washing processes and wait for it to finish (which
    takes ``washtime`` minutes).
    """

    def __init__(self, env, num_machines, washtime): #constructor for the resource
        self.env = env
        self.machine = simpy.Resource(env, num_machines) #here the simpy.Resource is created
        self.washtime = washtime

    def wash(self, car): # this is the wash method that the resource supports
        yield self.env.timeout(self.washtime)
        pct_dirt = random.randint(50, 99)
        print(f"Carwash removed {pct_dirt}% of {car}'s dirt.")

### Step 2: Entity (`Car` class)

Entities are the **things that move through the system** (customers, orders, cars, ...). Their **flow** — arrive, wait, get served, leave — is modeled as a method within the entity class:

In [3]:
class Car:  #define a separate class for the entity
    
    def __init__(self, env, name): #constructor for the entity
        self.env = env
        self.name = name

    def run(self, carwash): #method to model the flow of entities through the system
        print(f'{self.name} arrives at the carwash at {self.env.now:.2f}.')
        with carwash.machine.request() as request: #for washing, one machine is requested
            yield request #we wait for the machine to be available

            print(f'{self.name} enters the carwash at {self.env.now:.2f}.') #the machine is now available
            yield self.env.process(carwash.wash(self.name)) #we start the washing process 

            print(f'{self.name} leaves the carwash at {self.env.now:.2f}.') # the car leaves the carwash

Note how the `run` method connects to the resource: it calls `carwash.machine.request()` to wait for a machine, then `carwash.wash()` to start the washing process. The entity **uses** the resource — this is how the two classes interact.

### Step 3: Entity Generator (`car_generator` function)

The **arrival of entities** is a separate process from their **flow** through the system. The generator creates entities and registers their flow as a process:

In [4]:
def car_generator(env, t_inter, carwash):
    car_count = 0

    # Create 4 initial cars
    for _ in range(4):
        car = Car(env, name=f'Car {car_count}')
        car_count += 1
        env.process(car.run(carwash))

    # Create more cars while the simulation is running
    while True:
        yield env.timeout(random.randint(t_inter - 2, t_inter + 2))
        car = Car(env, name=f'Car {car_count}')
        car_count += 1
        env.process(car.run(carwash))

### Step 4: Run Simulation

To run a simulation, always follow these five steps:

1. Define parameters
2. Create the SimPy environment (`simpy.Environment()`)
3. Create the resources
4. Start the entity generation process
5. Execute the simulation (`env.run`)

In [5]:
# 1. Define parameters
RANDOM_SEED = 42
NUM_MACHINES = 2  # Number of machines in the carwash
WASHTIME =  5     # Minutes it takes to clean a car
T_INTER = 7       # Create a car every ~T_INTER minutes
SIM_TIME = 50     # Simulation time in minutes


# 2. Create an environment and start the setup process
env = simpy.Environment()

# 3. create resources
carwash = Carwash(env, NUM_MACHINES, WASHTIME)

# 4. call the process to generate the entities
env.process(car_generator(env, T_INTER, carwash))

# 5. Execute simulation
print('Running Simulation...')
random.seed(RANDOM_SEED)  # This helps to reproduce the results (more on this later)
env.run(until=SIM_TIME)
print('... Done')

Running Simulation...
Car 0 arrives at the carwash at 0.00.
Car 1 arrives at the carwash at 0.00.
Car 2 arrives at the carwash at 0.00.
Car 3 arrives at the carwash at 0.00.
Car 0 enters the carwash at 0.00.
Car 1 enters the carwash at 0.00.
Car 4 arrives at the carwash at 5.00.
Carwash removed 97% of Car 0's dirt.
Carwash removed 67% of Car 1's dirt.
Car 0 leaves the carwash at 5.00.
Car 1 leaves the carwash at 5.00.
Car 2 enters the carwash at 5.00.
Car 3 enters the carwash at 5.00.
Car 5 arrives at the carwash at 10.00.
Carwash removed 64% of Car 2's dirt.
Carwash removed 58% of Car 3's dirt.
Car 2 leaves the carwash at 10.00.
Car 3 leaves the carwash at 10.00.
Car 4 enters the carwash at 10.00.
Car 5 enters the carwash at 10.00.
Carwash removed 97% of Car 4's dirt.
Carwash removed 56% of Car 5's dirt.
Car 4 leaves the carwash at 15.00.
Car 5 leaves the carwash at 15.00.
Car 6 arrives at the carwash at 16.00.
Car 6 enters the carwash at 16.00.
Carwash removed 55% of Car 6's dirt.
Ca

## From Printing to Data Collection

Printing simulation events is useful for debugging, but to actually **answer questions** (like "what is the average waiting time?"), we need to **collect data** systematically. 

The key idea: **separate data collection from the simulation logic.** We introduce a dedicated `EventLogger` class that records events. This keeps the simulation code clean — entities just call the logger, and analysis happens afterwards.

In [6]:
import pandas as pd

class EventLogger:
    def __init__(self):
        self.logs = []
    
    def log(self, **kwargs):
        """Record one event. Pass any keyword arguments you want to track."""
        self.logs.append(kwargs)
    
    def get_df(self):
        """Convert all logged events into a pandas DataFrame."""
        return pd.DataFrame(self.logs)

**How does `**kwargs` work?** The `log` method uses `**kwargs` — the double-asterisk operator for **dictionary unpacking**. It accepts any number of named arguments and packs them into a dictionary:

```python
logger.log(car="Car 0", arrival_time=0, waiting_time=5)
# internally, kwargs becomes: {'car': 'Car 0', 'arrival_time': 0, 'waiting_time': 5}
```

This is what makes the logger generic — you don't have to define in advance which fields to log. You can pass different fields for different models or even different event types within the same simulation.

**How does `get_df()` work?** Internally, `self.logs` is a list of dictionaries — one per logged event. `pd.DataFrame(...)` converts this list into a table where each dictionary becomes a row and each key becomes a column:

```python
logger.logs
# [{'car': 'Car 0', 'arrival_time': 0, 'waiting_time': 5},
#  {'car': 'Car 1', 'arrival_time': 3, 'queue_length': 1}]

logger.get_df()
#       car  arrival_time  waiting_time  queue_length
# 0  Car 0             0           5.0           NaN
# 1  Car 1             3           NaN           1.0
```

If different events have different keys, pandas fills the missing values with `NaN` — so you can safely log different event types into the same logger.

We modify the `Car` class to use the logger. Instead of printing, the entity calls `self.logger.log(...)` at the end of its flow. The `Carwash` class remains unchanged.

In [7]:
class Car:
    
    def __init__(self, env, name, logger):
        self.env = env
        self.name = name
        self.logger = logger  # shared EventLogger instance

    def run(self, carwash):
        arrival_time = self.env.now
        queue_length = len(carwash.machine.queue)
        
        with carwash.machine.request() as request:
            yield request
            waiting_time = self.env.now - arrival_time

            yield self.env.process(carwash.wash(self.name))
            
            # Log the event instead of (only) printing
            self.logger.log(
                car=self.name,
                arrival_time=arrival_time,
                queue_length=queue_length,
                waiting_time=waiting_time,
            )

The generator needs a small update to pass the `logger` to each car:

In [8]:
def car_generator(env, t_inter, carwash, logger):
    car_count = 0

    # Create 4 initial cars
    for _ in range(4):
        car = Car(env, name=f'Car {car_count}', logger=logger)
        car_count += 1
        env.process(car.run(carwash))

    # Create more cars while the simulation is running
    while True:
        yield env.timeout(random.randint(t_inter - 2, t_inter + 2))
        car = Car(env, name=f'Car {car_count}', logger=logger)
        car_count += 1
        env.process(car.run(carwash))

In [9]:
# Run simulation with data collection
RANDOM_SEED = 42
NUM_MACHINES = 2
WASHTIME = 5
T_INTER = 7
SIM_TIME = 50

random.seed(RANDOM_SEED)
logger = EventLogger()  # create a logger to collect data

env = simpy.Environment()
carwash = Carwash(env, NUM_MACHINES, WASHTIME)
env.process(car_generator(env, T_INTER, carwash, logger))
env.run(until=SIM_TIME)

Carwash removed 97% of Car 0's dirt.
Carwash removed 67% of Car 1's dirt.
Carwash removed 64% of Car 2's dirt.
Carwash removed 58% of Car 3's dirt.
Carwash removed 97% of Car 4's dirt.
Carwash removed 56% of Car 5's dirt.
Carwash removed 55% of Car 6's dirt.
Carwash removed 77% of Car 7's dirt.
Carwash removed 55% of Car 8's dirt.
Carwash removed 64% of Car 9's dirt.
Carwash removed 82% of Car 10's dirt.


Now the logger contains all recorded events. We retrieve them as a DataFrame:

In [10]:
df = logger.get_df()
df

,car,arrival_time,queue_length,waiting_time
0,Car 0,0,0,0
1,Car 1,0,0,0
2,Car 2,0,0,5
3,Car 3,0,1,5
4,Car 4,5,2,5
5,Car 5,10,1,0
6,Car 6,16,0,0
7,Car 7,25,0,0
8,Car 8,34,0,0
9,Car 9,39,0,0


### Analyzing the Results

With the data in a DataFrame, we can answer questions about the system:

In [11]:
from lets_plot import *
LetsPlot.setup_html()

# Waiting time per car
(ggplot(df, aes(x='car', y='waiting_time'))
 + geom_bar(stat='identity', fill='steelblue')
 + labs(x='Car', y='Waiting time (min)', title='Waiting Time per Car')
 + theme(axis_text_x=element_text(angle=45, hjust=1))
)

In [12]:
# Queue length at arrival
(ggplot(df, aes(x='arrival_time', y='queue_length'))
 + geom_step(color='tomato')
 + labs(x='Time (min)', y='Queue length at arrival', title='Queue Length Over Time')
)

In [13]:
print(f"Mean waiting time: {df['waiting_time'].mean():.2f} min")
print(f"Max queue length:  {df['queue_length'].max()}")

Mean waiting time: 1.36 min
Max queue length:  2


### The Pattern

The data collection pattern is always the same, regardless of the simulation:

1. Define an `EventLogger` class with `log()` and `get_df()` methods — **once, reuse everywhere**
2. Create a logger instance before the simulation: `logger = EventLogger()`
3. Inside the entity's flow method, call `self.logger.log(...)` with the relevant data
4. After the simulation, retrieve the results: `logger.get_df()`

The logger **separates data collection from simulation logic**. The entity doesn't need to know how data is stored or analyzed — it just reports what happened. This makes the simulation code cleaner and the logger reusable across different models.

## Summary

- **OOP basics:** Classes bundle data (attributes) and behavior (methods). `__init__` is the constructor, `self` refers to the current instance.
- **Modeling conventions:** Entity class (flow as method), resource class (service as method), generator function (arrivals), EventLogger (data collection).
- **Carwash example:** Built a full simulation following the conventions — resource (`Carwash`), entity (`Car`), generator (`car_generator`), 5-step run pattern.
- **Data collection:** The `EventLogger` separates logging from simulation logic. Entities call `logger.log(...)`, results are retrieved via `logger.get_df()`.
- **`**kwargs`:** The double-asterisk operator makes the logger generic — any keyword arguments are packed into a dictionary.

## Next Lecture

Condition events and more advanced process interactions.

### Further Materials

- **OOP in Python:** We provide a free DataCamp course on object-oriented programming. Sign up [here](https://www.datacamp.com/groups/shared_links/4be874d860629f7903a9aa91acb3c7e0c117d8341af65b1a2a1ba82442a19c6f) (you must use your university email address).
- **pandas:** We also provide a free DataCamp course on data manipulation with pandas. Sign up [here](https://www.datacamp.com/groups/shared_links/4be874d860629f7903a9aa91acb3c7e0c117d8341af65b1a2a1ba82442a19c6f) (you must use your university email address).
- **lets-plot:** We provide a notebook on the basics of lets-plot on WueCampus.